# IMF JSON → Excel Dataset

This workflow includes both Structural Benchmarks and Prior Actions.
The output files below capture both datasets, so check the PA_Master sheet as well as the SB sheets.

**Setup (one-time):** copy `.env.example` to `.env` and fill in `BASE_FOLDER` and
`EXCLUDE_LIST_PATH`. Run cells 1-11 to define helpers, then run **Cell 12** to process
all countries and save per-country + aggregate Excel/JSON under `BASE_FOLDER/_Aggregate`.


In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"])

import os
from dotenv import load_dotenv

# Loads BASE_FOLDER, EXCLUDE_LIST_PATH from .env (see .env.example; .env is git-ignored)
load_dotenv()

BASE_FOLDER    = os.environ.get("BASE_FOLDER", "")
COUNTRY_FILTER = ['GHA']  # None = all │ "BEN" = one │ ["BEN","TZA"] = subset
LATEST_ONLY    = True
EXCLUDE_LIST   = os.environ.get("EXCLUDE_LIST_PATH", "")

PIPELINE_SCOPE = "Structural Benchmarks (SB / Reform Target), Prior Actions and QPC"
print("NOTE: this workbook includes Structural Benchmarks (with a Type column: SB / Reform Target), Prior Actions and QPC.")

if not BASE_FOLDER:
    raise ValueError("Set BASE_FOLDER in your .env file.")
if not EXCLUDE_LIST:
    raise ValueError("Set EXCLUDE_LIST_PATH in your .env file.")
# ──────────────────────────────────────────────────────────────────────────────


NOTE: this workbook is for Structural Benchmarks only; Prior Actions are intentionally excluded.



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip


In [30]:
import pandas as pd

In [31]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "openpyxl", "tqdm", "python-dotenv", "-q"])



[notice] A new release of pip is available: 25.3 -> 26.2.2
[notice] To update, run: pip3 install --upgrade pip


CompletedProcess(args=['/usr/local/bin/python3', '-m', 'pip', 'install', 'openpyxl', 'tqdm', 'python-dotenv', '-q'], returncode=0)

In [32]:
import json, re, glob
from pathlib import Path
from openpyxl import Workbook

# Load exclude list from file (so you can update it without editing the notebook)
import openpyxl as _opx

def _load_exclude_set(path, sheet_name="Box_Annex_Include"):
    """Returns the set of base_names to EXCLUDE (standard/boilerplate IMF
    annexes and boxes). Understands two schemas:
      - base_name/count/Remove lookup table  -> exclude where Remove == 1
      - per-row base_name/Include export      -> exclude where Include == 0
    """
    _wb = _opx.load_workbook(path)
    if sheet_name in _wb.sheetnames:
        _sheet = _wb[sheet_name]
    else:
        _sheet = _wb.worksheets[0]
        print(f"Warning: sheet '{sheet_name}' not found in {Path(path).name} "
              f"— using first sheet '{_sheet.title}' instead. Check EXCLUDE_LIST "
              f"points at the right file.")
    header = [str(c).strip() if c else "" for c in next(_sheet.iter_rows(max_row=1, values_only=True))]

    if "base_name" in header and "Remove" in header:
        ni, fi = header.index("base_name"), header.index("Remove")
        return {
            str(row[ni]).strip()
            for row in _sheet.iter_rows(min_row=2, values_only=True)
            if row[ni] and row[fi] == 1
        }
    elif "base_name" in header and "Include" in header:
        ni, fi = header.index("base_name"), header.index("Include")
        return {
            str(row[ni]).strip()
            for row in _sheet.iter_rows(min_row=2, values_only=True)
            if row[ni] and row[fi] == 0
        }
    else:
        raise ValueError(
            f"'{_sheet.title}' in {Path(path).name} has columns {header} — "
            f"expected 'base_name' plus either 'Remove' (1 = exclude) or "
            f"'Include' (0 = exclude). Check EXCLUDE_LIST points at the right file."
        )

_TOC_EXCLUDE_SET = _load_exclude_set(EXCLUDE_LIST)
print(f"Loaded {len(_TOC_EXCLUDE_SET)} exclusions from {Path(EXCLUDE_LIST).name}")


Loaded 132 exclusions from box_annex_exclude_list.xlsx


In [33]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def infer_doc_type(fname):
    f = fname.upper()

    # Selected Issues
    if re.search(r"[_\-\d]SI[_\-\d.]|[_\-]SI$|_SI\.PDF$", f):       return "Selected Issues"
    if "SELECTEDISSUES" in f.replace("_","").replace("-",""):         return "Selected Issues"
    if re.search(r"SELECTED.ISSUES?", f):                             return "Selected Issues"

    # Program Request
    if "REQUEST" in f:                                                 return "Program Request"

    # Check for Article IV marker (used in Review combos below)
    has_aiv = bool(re.search(r"ART(ICLE)?[_\-]?IV|ARTIV|_AIV[_.]|[_\-]AIV$", f))

    # Review — ordinal words first (avoids accidentally matching year digits)
    _ordinals = [("FIRST","1"),("SECOND","2"),("THIRD","3"),("FOURTH","4"),
                 ("FIFTH","5"),("SIXTH","6"),("SEVENTH","7"),("EIGHTH","8"),
                 ("NINTH","9"),("TENTH","10"),
                 ("1ST","1"),("2ND","2"),("3RD","3"),("4TH","4"),("5TH","5"),
                 ("6TH","6"),("7TH","7"),("8TH","8"),("9TH","9"),("10TH","10")]
    for word, num in _ordinals:
        if re.search(rf"{word}[_\-\s]?REV(IEW)?", f):
            return f"Review {num} + Art. IV" if has_aiv else f"Review {num}"

    # Review — digit before REV/REVIEW
    m = re.search(r"(\d+)[_\-]?REV(IEW)?", f)
    if m:
        n = m.group(1)
        if not re.match(r"^(19|20)\d{2}$", n):   # exclude years like 2023
            return f"Review {n} + Art. IV" if has_aiv else f"Review {n}"

    # Review — REV/REVIEW then digit
    m2 = re.search(r"REV(IEW)?[_\-]?(\d+)", f)
    if m2:
        n = m2.group(2)
        if not re.match(r"^(19|20)\d{2}$", n):
            return f"Review {n} + Art. IV" if has_aiv else f"Review {n}"

    # Plain REVIEW with no number
    if "REVIEW" in f:
        return "Review ? + Art. IV" if has_aiv else "Review ?"

    # Article IV standalone
    if has_aiv or "CONSULTATION" in f:                                return "Article IV"

    # Staff Report / DSA
    if "STAFF" in f and "REPORT" in f:                                return "Staff Report"
    if re.search(r"\bDSA\b|DEBT.SUST", f):                           return "DSA"

    return "Other"


_HEADER_ROW_RE = re.compile(
    r"^[\s|/]*(structural\s+benchmark|reform\s+area|due\s+date|status|"
    r"prior\s+actions?|benchmark\s+action|policy\s+action|proposed\s+action|"
    r"measures?|target\s+date|implementation\s+date|action|date|area|type)"
    r"([\s|/]+(structural\s+benchmark|reform\s+area|due\s+date|status|"
    r"prior\s+actions?|benchmark\s+action|policy\s+action|proposed\s+action|"
    r"measures?|target\s+date|implementation\s+date|action|date|area|type))*[\s|/]*$",
    re.IGNORECASE
)

def normalise(r):
    benches = []
    for tbl in r.get("benchmarks", []):
        clean_rows = {}
        for bt, vals in tbl.get("rows", {}).items():
            clean_vals = {}
            for col, v in vals.items():
                short = col.split(".")[-1].strip() if "." in col else col
                if re.search(r"due.?date|target.?date", short, re.I): short = "Due date"
                elif re.search(r"^status$", short, re.I):              short = "Status"
                elif re.search(r"reform.?area", short, re.I):          short = "Reform Area"
                elif re.search(r"structural.?benchmark", short, re.I): short = "Benchmark"
                clean_vals[short] = v
            bench_action = clean_vals.pop("Benchmark", None)
            display_text = bench_action if bench_action and len(bench_action) > len(bt) else bt
            clean_rows[display_text] = clean_vals
        benches.append({"table_title": tbl.get("table_title",""),
                        "page_num":    tbl.get("page_num",""),
                        "rows":        clean_rows})
    r2 = dict(r)
    r2["benchmarks"] = benches
    if not r2.get("doc_type"):
        r2["doc_type"] = infer_doc_type(r["file"])
    if r2["doc_type"] == "Selected Issues" and not r2.get("si_title"):
        r2["si_title"] = r["file"].replace(".pdf","").replace(".PDF","").replace("_"," ")
    return r2

def dedup(bench_list):
    seen = {}
    for tbl in bench_list:
        for bt, vals in tbl["rows"].items():
            bt = bt.strip()
            if not bt or _HEADER_ROW_RE.match(bt) or len(bt) < 15:
                continue
            nn = sum(1 for v in vals.values() if v)
            # Parse page numbers for comparison (handle non-numeric gracefully)
            def _page_num(entry):
                try:
                    return int(entry["page_num"])
                except (ValueError, TypeError):
                    return -1
            try:
                current_page = int(tbl["page_num"])
            except (ValueError, TypeError):
                current_page = -1
            if bt not in seen or current_page > _page_num(seen[bt]):
                seen[bt] = {"vals": vals, "nn": nn,
                             "table_title": tbl["table_title"],
                             "page_num":    tbl["page_num"]}
    return seen

def si_chapters(r):
    chapters = r.get("si_chapters")
    if chapters and isinstance(chapters, list):
        return [c for c in chapters if c and c.strip()]
    raw = r.get("si_title") or ""
    return [t.strip() for t in raw.split(" | ") if t.strip()]

def row_vals(r):
    dd = r.get("document_date") or {}
    country = r.get("subfolder","").strip("./") or r["file"].split("_")[0]
    return (country, dd.get("year"), dd.get("month"),
            dd.get("report_number"), dd.get("raw"),
            r.get("doc_type",""),
            r.get("si_title","") if r.get("doc_type")=="Selected Issues" else "")

# ── TOC exclusion list — load directly so this cell is self-contained ─────────
try:
    _TOC_EXCLUDE = _load_exclude_set(EXCLUDE_LIST)
    print(f"Loaded {len(_TOC_EXCLUDE)} TOC exclusions")
except Exception as _e:
    _TOC_EXCLUDE = set()
    print(f"Warning: could not load exclude list: {_e}")

def toc_base_name(title):
    """Strip any leading numbering/lettering prefix and trailing parenthetical
    to get the canonical base name for exclusion matching.
    Handles: "Box 1.", "Annex II.", "IV.", "V.", "3.", "A.", "A1.", "1.2."
             Trailing: "(RAM)", "(Concluded)", "(II)", "(cont.)"
    """
    t = str(title).strip()
    # 1. Strip "Box 1." / "Annex II." with category word
    t = re.sub(
        r"^(box|annex|figure|table|appendix|attachment)\s*"
        r"[\dIVXLCDMivxlcdm]+[\.:,\s]+",
        "", t, flags=re.IGNORECASE
    ).strip()
    # 2. Strip ANY leading numbering: "IV.", "3.", "A.", "A1.", "1.2."
    t = re.sub(
        r"^("
        r"[IVXLCDM]+\."          # Roman numerals: IV. VII. IX.
        r"|[A-Z]\d*[\.]"        # Letter: A. B1.
        r"|\d+(\.\d+)*[\.]"   # Digits: 1. 12. 1.2.
        r")\s+",
        "", t
    ).strip()
    # 3. Strip ANY trailing parenthetical: "(RAM)", "(Concluded)", "(II)", "(cont.)"
    t = re.sub(r"\s*\([^)]{1,30}\)\s*$", "", t).strip()
    return t

def toc_is_excluded(title):
    """Return True if this TOC entry should be filtered out."""
    return toc_base_name(title).lower() in {t.lower() for t in _TOC_EXCLUDE}

# ── SB new/newly-proposed filter ──────────────────────────────────────────────
_NEW_SB_RE = re.compile(
    r"newly\s+proposed|new\s+SB|newly\s+introduced|^new$|^proposed$|"
    r"not\s+yet\s+due",
    re.IGNORECASE
)

def is_new_sb(status):
    """Return True if the SB is new/newly proposed (no prior status = newly proposed)."""
    if not status:
        return True
    return bool(_NEW_SB_RE.search(str(status).strip()))

print("✓ Helpers ready")

Loaded 132 TOC exclusions
✓ Helpers ready


In [34]:
def build_workbook(results):
    # ── Sheet 1 ──────────────────────────────────────────────────
    # ── Sheet 1: Master ──────────────────────────────────────────────────────
    wb = Workbook()
    ws = wb.active; ws.title = "Master"
    ws.append(["Country","Year","Month","Report No","Doc Type","Date (raw)",
               "TOC Type","Chapter / TOC Title","TOC Page",
               "Reform Area","Benchmark Action","Due Date","SB Status","SB Type","SB Table","SB Page",
               "File"])
    
    for r in results:
        country, year, month, rep_no, date_raw, doc_type, si_title = row_vals(r)
        is_si    = doc_type == "Selected Issues"
        toc_list = r.get("toc", [])  # unfiltered — Master shows all boxes/annexes
        benches  = dedup(r.get("benchmarks", []))
        def toc_type(e): return "SI" if is_si else e["category"]
        if is_si:
            for idx, chapter in enumerate(si_chapters(r), 1):
                ws.append([country, year, month, rep_no, doc_type, date_raw,
                           "SI Chapter", chapter, idx,
                           "", "", "", "", "", "", "", r["file"]])
        else:
            for toc in toc_list:
                ws.append([country, year, month, rep_no, doc_type, date_raw,
                           toc_type(toc), toc["title"], toc["page"],
                           "", "", "", "", "", "", "", r["file"]])
        for bt, meta in benches.items():  # all statuses, no filtering
            vals = meta["vals"]
            ws.append([country, year, month, rep_no, doc_type, date_raw,
                       "Benchmark", "", "",
                       vals.get("Reform Area") or "", bt,
                       vals.get("Due date") or vals.get("Due_date") or "",
                       vals.get("Status") or "",
                       vals.get("Type") or "SB",
                       meta.get("table_title",""), meta.get("page_num",""), r["file"]])
        if not is_si and not toc_list and not benches:
            ws.append([country, year, month, rep_no, doc_type, date_raw,
                       "", "", "", "", "", "", "", "", "", "", r["file"]])
    
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    print(f"Sheet 1 Master: {ws.max_row - 1} rows")

    # ── Sheet 7: Benchmark_Dedup ──────────────────────────────────────────────
    # Deduplicated benchmarks.
    # For Review N documents, blank-status benchmarks are resolved by looking
    # back through Reviews N-1, N-2, … and the Program Request (norm50 key,
    # per-country):
    #   • Found in a prior document with a non-blank status → leave blank
    #     (the SB existed before; a blank here means status is pending/unknown)
    #   • Not found in any prior document → mark as "New"
    # For Program Request documents: blank status → always "New".

    # ── 7a. Build per-country existence index (norm50, any prior status) ─────
    # review_order: "Program Request" → 0, "Review N" → N, others → None
    def _review_order(doc_type):
        if doc_type == "Program Request":
            return 0
        m = re.match(r"Review\s+(\d+)", str(doc_type))
        return int(m.group(1)) if m else None

    def _norm50(bt):
        return re.sub(r"\s+", " ", bt.strip().lower())[:50]

    # {country: {norm50: min_order}}  — earliest order at which this SB appeared
    # (we only need to know *whether* it existed before, not which status)
    _sb_first_seen = {}
    for r in results:
        country, year, month, rep_no, date_raw, doc_type, si_title = row_vals(r)
        order = _review_order(doc_type)
        if order is None:
            continue
        for bt in dedup(r.get("benchmarks", [])):
            key = _norm50(bt)
            country_map = _sb_first_seen.setdefault(country, {})
            if key not in country_map or order < country_map[key]:
                country_map[key] = order

    def _resolve_blank_status(country, bt, current_order):
        """For a blank-status benchmark in Review current_order, return:
           - "New"  if the SB never appeared in any prior document
           - ""                   (leave blank) if it appeared in a prior document
        """
        key = _norm50(bt)
        first = _sb_first_seen.get(country, {}).get(key)
        if first is None or first >= current_order:
            return "New"
        return ""   # existed before → leave blank

    # ── 7b. Write the sheet ──────────────────────────────────────────────────
    ws7 = wb.create_sheet("SB_interim")
    ws7.append(["Country", "Year", "Month", "Report No", "Doc Type", "Date (raw)",
                "Reform Area", "Benchmark Action", "Due Date", "Status", "Type",
                "SB Table", "Page", "File"])

    for r in results:
        country, year, month, rep_no, date_raw, doc_type, si_title = row_vals(r)
        order = _review_order(doc_type)
        is_request = doc_type == "Program Request"

        deduped = dedup(r.get("benchmarks", []))

        for bt, meta in deduped.items():
            vals = meta["vals"]
            status = (vals.get("Status") or "").strip()

            if not status:
                if is_request:
                    status = "New"
                elif order is not None:
                    status = _resolve_blank_status(country, bt, order)
                # else: unknown doc type — leave blank

            ws7.append([
                country, year, month, rep_no, doc_type, date_raw,
                vals.get("Reform Area") or "",
                bt,
                vals.get("Due date") or vals.get("Due_date") or "",
                status,
                vals.get("Type") or "SB",
                meta.get("table_title", ""), meta.get("page_num", ""),
                r["file"],
            ])

    ws7.freeze_panes = "A2"
    ws7.auto_filter.ref = ws7.dimensions
    print(f"Sheet 7 SB_interim: {ws7.max_row - 1} rows")

    # ── Sheet 8: Benchmark_New_Modified ─────────────────────────────────────────
    # "New", "Continuous", and "Modified" SBs — same look-back logic as Sheet 7
    # for resolving blank statuses.
    # A benchmark is "New" if it was not present in any prior document
    # (for Review N) or if it is from a Program Request with no status.
    # "Continuous" and "Modified" benchmarks are kept as-is whenever Claude
    # returned that status verbatim, e.g.:
    #   "Continuous structural benchmark"
    #   "Modified continuous structural benchmark"
    #   "Modified"
    #   "Target date modified from January 2025"
    _NEW_STATUS_RE = re.compile(
        r"^new$|newly\s+proposed(?:\s+SB)?|new\s+SB|^proposed$",
        re.IGNORECASE
    )
    _CONTINUOUS_OR_MODIFIED_RE = re.compile(
        r"\bcontinuous\b|\bmodified\b",
        re.IGNORECASE
    )
    ws8 = wb.create_sheet("SB_New_Final")
    ws8.append(["Country", "Year", "Month", "Report No", "Doc Type", "Date (raw)",
                "Reform Area", "Benchmark Action", "Due Date", "Status", "Type",
                "SB Table", "Page", "File"])
    for r in results:
        country, year, month, rep_no, date_raw, doc_type, si_title = row_vals(r)
        order = _review_order(doc_type)
        is_request = doc_type == "Program Request"
        for bt, meta in dedup(r.get("benchmarks", [])).items():
            vals = meta["vals"]
            status = (vals.get("Status") or "").strip()
            if not status:
                if is_request:
                    status = "New"
                elif order is not None:
                    status = _resolve_blank_status(country, bt, order)
            if not (_NEW_STATUS_RE.search(status) or _CONTINUOUS_OR_MODIFIED_RE.search(status)):
                continue
            ws8.append([
                country, year, month, rep_no, doc_type, date_raw,
                vals.get("Reform Area") or "", bt,
                vals.get("Due date") or vals.get("Due_date") or "",
                status,
                vals.get("Type") or "SB",
                meta.get("table_title", ""), meta.get("page_num", ""),
                r["file"],
            ])
    ws8.freeze_panes = "A2"
    ws8.auto_filter.ref = ws8.dimensions
    print(f"Sheet 8 SB_New_Final: {ws8.max_row - 1} rows")


    # ── Sheet 2 ──────────────────────────────────────────────────
    # ── Sheet 2: Documents ────────────────────────────────────────────────────────
    ws = wb.create_sheet("Document_List")
    ws.append(["File","Country","Year","Month","Report No","Doc Type",
               "Date (raw)","SI Title","# Boxes/Chapters","# Annexes","# New SBs"])
    for r in results:
        country, year, month, rep_no, date_raw, doc_type, si_title = row_vals(r)
        toc = [e for e in r.get("toc", []) if not toc_is_excluded(e["title"])]
        is_si = doc_type == "Selected Issues"
        new_sbs = sum(1 for t in r.get("benchmarks",[]) for s in t["rows"].values()
                      if (not isinstance(s, dict) or (s.get("Type") or "SB") == "SB") and is_new_sb((s if isinstance(s,str) else s.get("Status","")) if isinstance(s,(str,dict)) else ""))
        ws.append([r["file"], country, year, month, rep_no, doc_type, date_raw, si_title,
                   len(si_chapters(r)) if is_si else sum(1 for e in toc if e["category"]=="Box"),
                   0 if is_si else sum(1 for e in toc if e["category"]=="Annex"),
                   new_sbs])
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    print(f"Sheet 2 Document_List: {ws.max_row - 1} rows")
    # ── Sheet 3 ──────────────────────────────────────────────────
    # ── Sheet 3: TOC_Filter ───────────────────────────────────────────────────────
    ws = wb.create_sheet("Box_Annex_Final")
    ws.append(["File","Country","Year","Month","Report No","Doc Type","TOC Type","TOC Title","Page"])
    for r in results:
        country, year, month, rep_no, _, doc_type, _ = row_vals(r)
        for e in r.get("toc", []):
            if toc_is_excluded(e["title"]): continue
            ws.append([r["file"], country, year, month, rep_no, doc_type,
                       e["category"], e["title"], e["page"]])
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    print(f"Sheet 3 Box_Annex_Final: {ws.max_row - 1} rows")
    # ── Sheet 3b: TOC_All ─────────────────────────────────────────────────────
    ws3b = wb.create_sheet("Box_Annex_All")
    ws3b.append(["File","Country","Year","Month","Report No","Doc Type","TOC Type","TOC Title","Page"])
    for r in results:
        country, year, month, rep_no, _, doc_type, _ = row_vals(r)
        for e in r.get("toc", []):
            ws3b.append([r["file"], country, year, month, rep_no, doc_type,
                         e["category"], e["title"], e["page"]])
    ws3b.freeze_panes = "A2"; ws3b.auto_filter.ref = ws3b.dimensions
    print(f"Sheet 3b Box_Annex_All: {ws3b.max_row - 1} rows")
    # ── Sheet 4 ──────────────────────────────────────────────────
    # ── Sheet 4: Benchmarks ──────────────────────────────────────────────────────
    ws = wb.create_sheet("SB_Master")
    ws.append(["File","Country","Year","Month","Report No","Doc Type",
               "Reform Area","Benchmark Action","Due Date","Status","Type","SB Table","Page"])
    for r in results:
        country, year, month, rep_no, _, doc_type, _ = row_vals(r)
        for tbl in r.get("benchmarks", []):
            for bt, vals in tbl["rows"].items():
                bt = bt.strip()
                if not bt or _HEADER_ROW_RE.match(bt) or len(bt) < 15:
                    continue
                ws.append([r["file"], country, year, month, rep_no, doc_type,
                           vals.get("Reform Area") or "", bt,
                           vals.get("Due date") or vals.get("Due_date") or "",
                           vals.get("Status") or "",
                           vals.get("Type") or "SB",
                           tbl.get("table_title",""), tbl.get("page_num","")])
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    print(f"Sheet 4 SB_Master: {ws.max_row - 1} rows")
    # ── Sheet 5 ──────────────────────────────────────────────────
    # ── Sheet 5: Selected_Issues ──────────────────────────────────────────────────
    ws = wb.create_sheet("SIP")
    ws.append(["Country","Year","Month","Report No",
               "Linked Main Report","Chapter #","Chapter Title","SI File"])
    # Keyed by (country, year) -- NOT year alone. Year-only keying let one
    # country's main report leak into every other country's SI rows for the
    # same calendar year (whichever country's report was inserted first won
    # the slot for that year, for all countries). Kept as a fallback only.
    main_by_country_year = {}
    for r in results:
        if r.get("doc_type") != "Selected Issues":
            yr = (r.get("document_date") or {}).get("year")
            country = r.get("subfolder","").strip("./") or r["file"].split("_")[0]
            key = (country, yr)
            if yr and key not in main_by_country_year:
                main_by_country_year[key] = r["file"]

    # Primary match: an SI's IMF Report No is always exactly one more than its
    # main report's (e.g. SI "24/339" accompanies main report "24/338" -- the
    # same number that appears in Master's "Report No" column). Build a
    # report_number -> (file, country) index across ALL documents, then for
    # each SI look up report_number - 1 within the same country.
    _REPORT_NO_RE = re.compile(r"^\s*(\d+)\s*/\s*(\d+)\s*$")

    def _prev_report_number(rn):
        m = _REPORT_NO_RE.match(rn or "")
        if not m:
            return None
        series, num = m.group(1), int(m.group(2))
        return f"{series}/{num - 1}" if num > 0 else None

    main_by_report_no = {}
    for r in results:
        rn = (r.get("document_date") or {}).get("report_number")
        country_r = r.get("subfolder","").strip("./") or r["file"].split("_")[0]
        if rn and rn not in main_by_report_no:
            main_by_report_no[rn] = (r["file"], country_r)

    for r in results:
        if r.get("doc_type") != "Selected Issues": continue
        dd = r.get("document_date") or {}
        year = dd.get("year")
        country = r.get("subfolder","").strip("./") or r["file"].split("_")[0]
        chapters = si_chapters(r)

        linked = ""
        target_rn = _prev_report_number(dd.get("report_number"))
        if target_rn and target_rn in main_by_report_no:
            cand_file, cand_country = main_by_report_no[target_rn]
            if cand_country == country:
                linked = cand_file
        if not linked:
            # Report-number match unavailable (missing/malformed report_number,
            # or the main report simply isn't in this batch) -- fall back to
            # the coarser same-country/same-year heuristic.
            linked = main_by_country_year.get((country, year), "")

        if chapters:
            for idx, chapter in enumerate(chapters, 1):
                ws.append([country, year, dd.get("month"), dd.get("report_number"),
                           linked, idx, chapter, r["file"]])
        else:
            ws.append([country, year, dd.get("month"), dd.get("report_number"),
                       linked, "", "(no chapters extracted)", r["file"]])
    ws.freeze_panes = "A2"; ws.auto_filter.ref = ws.dimensions
    print(f"Sheet 5 SIP: {ws.max_row - 1} rows")
    # ── Sheet 6 ──────────────────────────────────────────────────
    # ── Sheet 6: SB_Tracker (benchmark × review round pivot) ─────────────────────
    ws = wb.create_sheet("SB_Tracker")
    
    docs_with_sb = [r for r in results if r.get("benchmarks")]
    col_labels = [
        f"{(r.get('document_date') or {}).get('year','?')} "
        f"{(r.get('document_date') or {}).get('month','')[:3]} "
        f"({(r.get('document_date') or {}).get('report_number','')})"
        for r in docs_with_sb
    ]
    
    # Collect all unique benchmarks: norm -> (reform_area, display_text, due_date)
    all_norms = {}
    for r in docs_with_sb:
        for bt, meta in dedup(r.get("benchmarks",[])).items():
            vals = meta["vals"]
            norm = re.sub(r"\s+", " ", bt.strip().lower())[:200]
            if norm not in all_norms:
                all_norms[norm] = (
                    vals.get("Reform Area") or "—",
                    bt.strip(),
                    vals.get("Due date") or vals.get("Due_date") or "",
                    vals.get("Type") or "SB",
                )
    
    sorted_norms = sorted(all_norms.items(), key=lambda x: (x[1][0], x[1][1]))
    
    # Status matrix: norm → {doc_index: status}
    status_mx = {norm: {} for norm in all_norms}
    for di, r in enumerate(docs_with_sb):
        for bt, meta in dedup(r.get("benchmarks",[])).items():
            norm = re.sub(r"\s+", " ", bt.strip().lower())[:200]
            if norm in status_mx:
                status_mx[norm][di] = meta["vals"].get("Status") or ""
    
    # Header row
    ws.append(["Reform Area", "Benchmark Action", "Due Date", "Type"] + col_labels)
    
    # Data rows
    prev_area = None
    for norm, (reform_area, display_text, due_date, sb_type) in sorted_norms:
        area_label = reform_area if reform_area != prev_area else ""
        statuses   = [status_mx[norm].get(di, "") for di in range(len(docs_with_sb))]
        ws.append([area_label, display_text, due_date, sb_type] + statuses)
        prev_area = reform_area
    
    ws.freeze_panes = "E2"
    ws.auto_filter.ref = ws.dimensions
    print(f"Sheet 6 SB_Tracker: {ws.max_row - 1} benchmark rows × {len(docs_with_sb)} review rounds")
    return wb

In [35]:
# Sheet 2 code is inside build_workbook() in Cell 5
pass


In [36]:
# Sheet 3 code is inside build_workbook() in Cell 5
pass


In [37]:
# Sheet 4 code is inside build_workbook() in Cell 5
pass


In [38]:
# Sheet 5 code is inside build_workbook() in Cell 5
pass


In [39]:
# Sheet 6 code is inside build_workbook() in Cell 5
pass


In [40]:
# Run Cell 12 to process all countries and save Excel + JSON outputs.
print("Ready — run Cell 12.")


Ready — run Cell 12.


In [41]:
# ── Multi-country mode ────────────────────────────────────────────────────────
# Processes all countries and saves per-country + aggregate Excel/JSON.
# Edit BASE_FOLDER / COUNTRY_FILTER / LATEST_ONLY in Cell 1, then run this.
from datetime import datetime
from tqdm import tqdm

def _load_country(folder, latest_only=True):
    files = sorted(glob.glob(str(folder / "toc_results_*.json")),
                   key=lambda f: Path(f).stat().st_mtime, reverse=True)  # newest by mod time
    if not files: return []
    if latest_only: files = files[:1]
    merged, seen = [], set()
    for fpath in files:
        d = json.load(open(fpath, encoding="utf-8"))
        rows = d.get("results") or list(d.values())
        for r in rows:
            if not isinstance(r, dict): continue
            key = (fpath, r.get("path") or r.get("file") or id(r))
            if key not in seen:
                seen.add(key)
                merged.append(r)
    tqdm.write(f"  {folder.name}: {len(merged)} docs")
    return merged

base = Path(BASE_FOLDER).expanduser().resolve()
agg_dir = base / "_Aggregate"; agg_dir.mkdir(exist_ok=True)
cf = COUNTRY_FILTER
if cf:
    cs = [cf] if isinstance(cf,str) else cf
    folders = [base/c for c in cs if (base/c).is_dir()]
else:
    folders = sorted([d for d in base.iterdir()
                      if d.is_dir() and d.name != "_Aggregate"
                      and glob.glob(str(d/"toc_results_*.json"))])

print(f"Found {len(folders)} folder(s): {[f.name for f in folders]}\n")
all_raw = []
for folder in tqdm(folders, desc="Countries", unit="country"):
    raw = _load_country(folder, latest_only=LATEST_ONLY)
    if not raw: continue
    nr = [normalise(r) for r in raw]
    build_workbook(nr).save(folder / f"IMF_{folder.name}.xlsx")
    json.dump({"generated":datetime.now().isoformat(),"country":folder.name,"total_docs":len(raw),"results":raw},
              open(folder/f"toc_aggregate_{folder.name}.json","w",encoding="utf-8"), indent=2, ensure_ascii=False)
    tqdm.write(f"  Saved IMF_{folder.name}.xlsx + toc_aggregate_{folder.name}.json")
    all_raw.extend(raw)

if all_raw:
    build_workbook([normalise(r) for r in all_raw]).save(agg_dir/"IMF_All_Countries.xlsx")
    json.dump({"generated":datetime.now().isoformat(),"countries":[f.name for f in folders],"total_docs":len(all_raw),"results":all_raw},
              open(agg_dir/"toc_aggregate_all.json","w",encoding="utf-8"), indent=2, ensure_ascii=False)
    print(f"\n✓ Aggregate: {agg_dir}")
print("\nDone.")


Found 1 folder(s): ['GHA']



Countries:   0%|          | 0/1 [00:00<?, ?country/s]

  GHA: 8 docs
Sheet 1 Master: 214 rows
Sheet 7 SB_interim: 166 rows
Sheet 8 SB_New_Final: 74 rows
Sheet 2 Document_List: 8 rows
Sheet 3 Box_Annex_Final: 24 rows
Sheet 3b Box_Annex_All: 48 rows
Sheet 4 SB_Master: 166 rows
Sheet 5 SIP: 0 rows
Sheet 6 SB_Tracker: 76 benchmark rows × 8 review rounds


Countries: 100%|██████████| 1/1 [00:00<00:00,  8.61country/s]

  Saved IMF_GHA.xlsx + toc_aggregate_GHA.json
Sheet 1 Master: 214 rows
Sheet 7 SB_interim: 166 rows
Sheet 8 SB_New_Final: 74 rows
Sheet 2 Document_List: 8 rows
Sheet 3 Box_Annex_Final: 24 rows
Sheet 3b Box_Annex_All: 48 rows
Sheet 4 SB_Master: 166 rows
Sheet 5 SIP: 0 rows
Sheet 6 SB_Tracker: 76 benchmark rows × 8 review rounds



✓ Aggregate: /Users/narminebenaissa/Downloads/pdf-section-extractor-main/IMF_Downloads_Renamed/_Aggregate

Done.


In [42]:
from pathlib import Path
import glob

base = Path(BASE_FOLDER).expanduser().resolve()
print("BASE_FOLDER resolves to:", base)
print("Exists:", base.exists())
print()

if base.exists():
    for d in sorted(base.iterdir()):
        if d.is_dir():
            matches = glob.glob(str(d / "toc_results_*.json"))
            print(f"{d.name:20s} | is_dir={d.is_dir()} | toc_results_*.json matches: {len(matches)}")

BASE_FOLDER resolves to: /Users/narminebenaissa/Downloads/pdf-section-extractor-main/IMF_Downloads_Renamed
Exists: True

.git                 | is_dir=True | toc_results_*.json matches: 0
BEN                  | is_dir=True | toc_results_*.json matches: 4
BFA                  | is_dir=True | toc_results_*.json matches: 2
CAF                  | is_dir=True | toc_results_*.json matches: 2
CIV                  | is_dir=True | toc_results_*.json matches: 1
CMR                  | is_dir=True | toc_results_*.json matches: 6
COD                  | is_dir=True | toc_results_*.json matches: 2
COG                  | is_dir=True | toc_results_*.json matches: 1
COM                  | is_dir=True | toc_results_*.json matches: 2
CPV                  | is_dir=True | toc_results_*.json matches: 3
ETH                  | is_dir=True | toc_results_*.json matches: 1
GHA                  | is_dir=True | toc_results_*.json matches: 2
GMB                  | is_dir=True | toc_results_*.json matches: 1
GNB     

In [ ]:
# Add a dedicated Prior-Action sheet to the workbook without rewriting the
# whole transformation function. This keeps the existing SB workflow intact while
# exposing the PA dataset in a clear sheet.
_original_build_workbook = build_workbook

def build_workbook(results):
    wb = _original_build_workbook(results)
    ws_pa = wb.create_sheet("PA_Master")
    ws_pa.append(["File", "Country", "Year", "Month", "Report No", "Doc Type",
                  "PA Table", "Page", "Category", "Reform Area", "Action", "Status"])
    for r in results:
        country, year, month, rep_no, date_raw, doc_type, si_title = row_vals(r)
        for tbl in r.get("prior_actions", []):
            for action, vals in tbl.get("rows", {}).items():
                if not isinstance(vals, dict):
                    continue
                ws_pa.append([
                    r["file"], country, year, month, rep_no, doc_type,
                    tbl.get("table_title", ""), tbl.get("page_num", ""),
                    vals.get("Category") or "Prior Action",
                    vals.get("Reform Area") or "",
                    action,
                    vals.get("Status") or "",
                ])
    ws_pa.freeze_panes = "A2"
    ws_pa.auto_filter.ref = ws_pa.dimensions
    print(f"Sheet PA_Master: {ws_pa.max_row - 1} rows")

    # ── QPC sheets (Quantitative Performance Criteria / Indicative Targets) ──
    # QPC_Master: long format, one row per criterion x test date x sub-column.
    # QPC_Wide:   one row per (document, criterion); one column per
    #             "<test date> — <sub-column>" (ordered as first seen).
    ws_q = wb.create_sheet("QPC_Master")
    ws_q.append(["File", "Country", "Year", "Month", "Report No", "Doc Type",
                 "QPC Table", "Page", "Unit Note", "Category", "Criterion",
                 "Test Date", "Sub-column", "Value"])
    wide_rows, col_order = [], []
    for r in results:
        country, year, month, rep_no, date_raw, doc_type, si_title = row_vals(r)
        for tbl in r.get("qpc", []):
            for crit in tbl.get("criteria", []):
                cells = {}
                for td in crit.get("test_dates") or []:
                    if not isinstance(td, dict):
                        continue
                    for sub, val in (td.get("values") or {}).items():
                        label = f"{td.get('date') or ''} — {sub}"
                        cells[label] = val
                        if label not in col_order:
                            col_order.append(label)
                        ws_q.append([
                            r["file"], country, year, month, rep_no, doc_type,
                            tbl.get("table_title", ""), tbl.get("page_num", ""),
                            tbl.get("unit_note") or "", crit.get("category") or "",
                            crit.get("criterion") or "", td.get("date") or "", sub, val,
                        ])
                wide_rows.append((
                    [r["file"], country, year, month, rep_no, doc_type,
                     tbl.get("table_title", ""), tbl.get("page_num", ""),
                     tbl.get("unit_note") or "", crit.get("category") or "",
                     crit.get("criterion") or ""], cells))
    ws_q.freeze_panes = "A2"
    ws_q.auto_filter.ref = ws_q.dimensions
    print(f"Sheet QPC_Master: {ws_q.max_row - 1} rows")

    ws_w = wb.create_sheet("QPC_Wide")
    ws_w.append(["File", "Country", "Year", "Month", "Report No", "Doc Type",
                 "QPC Table", "Page", "Unit Note", "Category", "Criterion"] + col_order)
    for base_cols, cells in wide_rows:
        ws_w.append(base_cols + [cells.get(c, "") for c in col_order])
    ws_w.freeze_panes = "L2"
    ws_w.auto_filter.ref = ws_w.dimensions
    print(f"Sheet QPC_Wide: {ws_w.max_row - 1} rows x {len(col_order)} value columns")
    return wb

print("Sheet hook installed; build_workbook() now includes PA_Master, QPC_Master, QPC_Wide.")
